# Inside Airbnb: Munich Listing Points and Text Descriptions

This notebook introduces **Inside Airbnb** as an example of open, tabular point data with text attributes. The Munich summary `listings.csv` file contains one row per Airbnb listing and includes latitude and longitude columns. The detailed `listings.csv.gz` file adds individual text fields such as listing descriptions.

## What This Notebook Does

1. Finds the current Munich download links from the official Inside Airbnb data page.
2. Reads the lightweight `visualisations/listings.csv` table for point locations.
3. Optionally joins individual listing descriptions from `data/listings.csv.gz`.
4. Converts `longitude` and `latitude` into a GeoDataFrame with point geometries.
5. Loads Munich neighbourhood polygons from `neighbourhoods.geojson`.
6. Creates quick summaries for room type, price, and neighbourhood counts.
7. Maps listing points with Folium; each popup includes listing-level text when available.
8. Demonstrates a spatial join between listing points and neighbourhood polygons.
9. Saves a prepared GeoJSON file that can be reused in later course notebooks.

## Why This Dataset Is Useful

Airbnb listings are easy to understand but analytically rich. They connect tabular attributes such as `room_type`, `price`, and `number_of_reviews` with unstructured text such as descriptions and neighbourhood overviews. This supports examples of spatial analysis, text preprocessing, feature engineering, and platform-data bias.

## Data Source

- Inside Airbnb Get the Data page: https://insideairbnb.com/get-the-data/
- Example city: Munich, Bavaria, Germany
- Data files used here: `listings.csv`, `listings.csv.gz`, and `neighbourhoods.geojson`
- License noted by Inside Airbnb: Creative Commons Attribution 4.0 International License

Inside Airbnb is a public research dataset, not an official housing registry. Treat the data as useful for teaching and exploratory analysis, but discuss scraping limitations, missing values, platform bias, and temporal snapshots.

In [ ]:
# Colab setup
!pip -q install geopandas shapely pyproj folium mapclassify beautifulsoup4 requests

In [ ]:
from pathlib import Path
from urllib.parse import urljoin
import html
import re

import folium
import geopandas as gpd
import pandas as pd
import requests
from bs4 import BeautifulSoup
from folium.plugins import MarkerCluster
from shapely.geometry import Point

## 1. Configuration

The notebook uses the lightweight **summary listings** file from Inside Airbnb:

- `visualisations/listings.csv`: one row per listing, including coordinates and selected listing attributes
- `visualisations/neighbourhoods.geojson`: Munich neighbourhood polygons for mapping and spatial joins

The detailed `listings.csv.gz` file contains many more columns, but the summary file is faster and easier for classroom use. The notebook writes derived outputs to Colab's temporary `/content/inside_airbnb_munich` directory.

In [ ]:
INSIDE_AIRBNB_DATA_PAGE = "https://insideairbnb.com/get-the-data/"
CITY_LABEL = "Munich"
CACHE_DIR = Path("/content/inside_airbnb_munich")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Fallback URLs from the current Inside Airbnb Munich release shown on the Get the Data page.
# The link finder below should normally pick the current URLs automatically.
FALLBACK_URLS = {
    "summary_listings": "https://data.insideairbnb.com/germany/bv/munich/2025-09-27/visualisations/listings.csv",
    "detailed_listings": "https://data.insideairbnb.com/germany/bv/munich/2025-09-27/data/listings.csv.gz",
    "neighbourhoods_geojson": "https://data.insideairbnb.com/germany/bv/munich/2025-09-27/visualisations/neighbourhoods.geojson",
}

## 2. Find Current Munich Download Links

Inside Airbnb updates city snapshots over time. Instead of relying only on a hard-coded URL, this cell reads the official **Get the Data** page and searches for the Munich section.

The helper tries to find:

- the summary `listings.csv` file for point mapping,
- the detailed `listings.csv.gz` file for individual listing descriptions,
- the `neighbourhoods.geojson` boundary file.

The detailed table is larger, but it contains text fields such as listing descriptions and neighbourhood overviews. If link discovery fails because the website layout changes, the notebook falls back to the known Munich release URLs above.

In [ ]:
def find_inside_airbnb_city_links(city_label=CITY_LABEL, page_url=INSIDE_AIRBNB_DATA_PAGE):
    response = requests.get(page_url, timeout=60)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")

    city_heading = None
    for heading in soup.find_all(["h2", "h3", "h4"]):
        if city_label.lower() in heading.get_text(" ", strip=True).lower():
            city_heading = heading
            break

    if city_heading is None:
        raise ValueError(f"Could not find city heading for {city_label!r}")

    links = {}
    for node in city_heading.find_all_next():
        if node.name in ["h2", "h3", "h4"] and node is not city_heading:
            break
        if node.name != "a":
            continue
        href = node.get("href") or ""
        text = node.get_text(" ", strip=True).lower()
        url = urljoin(page_url, href)
        url_lower = url.lower()

        if text == "listings.csv" and "/visualisations/" in url_lower:
            links["summary_listings"] = url
        elif text == "neighbourhoods.geojson" and "/visualisations/" in url_lower:
            links["neighbourhoods_geojson"] = url
        elif text == "listings.csv.gz" and "/data/" in url_lower:
            links["detailed_listings"] = url

    missing = {"summary_listings", "neighbourhoods_geojson"} - set(links)
    if missing:
        raise ValueError(f"Missing expected links for {city_label}: {sorted(missing)}")
    return links


try:
    airbnb_links = find_inside_airbnb_city_links()
except Exception as exc:
    print(f"Using fallback URLs because link discovery failed: {exc}")
    airbnb_links = FALLBACK_URLS.copy()

airbnb_links

## 3. Read Listing Points

The summary listings table already has `latitude` and `longitude` columns. The loader below performs the minimum cleaning needed for geospatial work:

- checks that the coordinate columns exist,
- converts coordinates to numeric values,
- drops rows without valid coordinates,
- parses a numeric `price_eur` column when a `price` column is present,
- creates a GeoDataFrame in WGS84 (`EPSG:4326`).

The resulting `gdf_listings` object can be mapped directly or joined to other spatial layers.

In [ ]:
def read_munich_listings(url=airbnb_links["summary_listings"]):
    df = pd.read_csv(url)
    required = {"id", "latitude", "longitude"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Listings table is missing required columns: {sorted(missing)}")

    df = df.dropna(subset=["latitude", "longitude"]).copy()
    df["latitude"] = pd.to_numeric(df["latitude"], errors="coerce")
    df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")
    df = df.dropna(subset=["latitude", "longitude"])

    if "price" in df.columns:
        df["price_eur"] = (
            df["price"].astype(str)
            .str.replace("€", "", regex=False)
            .str.replace("$", "", regex=False)
            .str.replace(",", "", regex=False)
            .str.strip()
        )
        df["price_eur"] = pd.to_numeric(df["price_eur"], errors="coerce")

    geometry = gpd.points_from_xy(df["longitude"], df["latitude"])
    return gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")


gdf_listings = read_munich_listings()
print(f"Munich listings with coordinates: {len(gdf_listings):,}")
gdf_listings.head()

## 4. Add Individual Listing Text

The lightweight summary table is best for mapping, but it may not contain full listing descriptions. The detailed `listings.csv.gz` table is larger and includes text columns such as `description`, `neighborhood_overview`, and sometimes `host_about`.

This step reads only a small set of text columns from the detailed table and joins them back to the point GeoDataFrame by `id`. The map popup can then show a short text introduction for each individual listing.

In [ ]:
TEXT_COLUMNS = [
    "id",
    "description",
    "neighborhood_overview",
    "host_about",
]


def clean_html_text(value):
    if pd.isna(value):
        return ""
    text = BeautifulSoup(str(value), "html.parser").get_text(" ")
    text = html.unescape(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def preview_text(value, max_chars=260):
    text = clean_html_text(value)
    if len(text) <= max_chars:
        return text
    return text[: max_chars - 1].rstrip() + "…"


def read_listing_texts(url=airbnb_links.get("detailed_listings")):
    if not url:
        print("No detailed listings URL available; keeping summary fields only.")
        return pd.DataFrame(columns=["id", "description_text", "neighborhood_overview_text", "host_about_text"])

    detailed = pd.read_csv(
        url,
        compression="infer",
        usecols=lambda col: col in TEXT_COLUMNS,
        low_memory=False,
    )
    if "id" not in detailed.columns:
        raise ValueError("Detailed listings table has no id column.")

    detailed = detailed.drop_duplicates(subset=["id"]).copy()
    for col in ["description", "neighborhood_overview", "host_about"]:
        if col in detailed.columns:
            detailed[f"{col}_text"] = detailed[col].map(clean_html_text)
        else:
            detailed[f"{col}_text"] = ""

    return detailed[["id", "description_text", "neighborhood_overview_text", "host_about_text"]]


listing_texts = read_listing_texts()
gdf_listings = gdf_listings.merge(listing_texts, on="id", how="left")
for col in ["description_text", "neighborhood_overview_text", "host_about_text"]:
    if col not in gdf_listings.columns:
        gdf_listings[col] = ""
    gdf_listings[col] = gdf_listings[col].fillna("")

print(f"Listings with description text: {(gdf_listings['description_text'] != '').sum():,}")
gdf_listings[["id", "name", "description_text", "neighborhood_overview_text"]].head()

## 5. Quick Data Checks

Before mapping or modelling, inspect the attribute table. These summaries help students understand what each point represents:

- `room_type`: type of Airbnb offer, such as entire apartment or private room,
- `price_eur`: parsed numeric price field when available,
- `neighbourhood`: the neighbourhood label included by Inside Airbnb,
- `description_text`: a cleaned, shortened text introduction from the detailed listing table.

These checks are deliberately simple. They are a good place to discuss missing values, extreme prices, text cleaning, and whether scraped platform data should be treated as ground truth.

In [ ]:
print("Columns:")
print(list(gdf_listings.columns))

if "room_type" in gdf_listings.columns:
    display(gdf_listings["room_type"].value_counts(dropna=False).to_frame("count"))

if "price_eur" in gdf_listings.columns:
    display(gdf_listings["price_eur"].describe().to_frame("price_eur"))

if "neighbourhood" in gdf_listings.columns:
    display(gdf_listings["neighbourhood"].value_counts().head(10).to_frame("listings"))

if "description_text" in gdf_listings.columns:
    text_stats = pd.Series({
        "listings_total": len(gdf_listings),
        "with_description_text": int((gdf_listings["description_text"] != "").sum()),
        "with_neighborhood_overview_text": int((gdf_listings["neighborhood_overview_text"] != "").sum()),
    }).to_frame("count")
    display(text_stats)
    display(gdf_listings[["name", "room_type", "neighbourhood", "description_text"]].head(10))

## 6. Read Neighbourhood Boundaries

The neighbourhood boundary file is a GeoJSON polygon layer. It gives spatial context for the listing points and supports aggregation or spatial joins.

The notebook keeps the boundaries in WGS84 (`EPSG:4326`) so they match the listing point coordinates and can be displayed directly in Folium.

In [ ]:
def read_munich_neighbourhoods(url=airbnb_links["neighbourhoods_geojson"]):
    neighbourhoods = gpd.read_file(url).to_crs("EPSG:4326")
    return neighbourhoods


gdf_neighbourhoods = read_munich_neighbourhoods()
print(f"Munich neighbourhood polygons: {len(gdf_neighbourhoods):,}")
gdf_neighbourhoods.head()

## 7. Map Listing Points

The map uses a marker cluster so that many listings remain readable in a browser. Points are colored by `room_type` when that field is available.

Each popup includes the listing name, room type, price, neighbourhood, and the full available individual text fields. The popup content has a fixed height and can be scrolled, so long descriptions remain available without making the map unusable.

In [ ]:
ROOM_TYPE_COLORS = {
    "Entire home/apt": "#1f77b4",
    "Private room": "#2ca02c",
    "Shared room": "#ff7f0e",
    "Hotel room": "#9467bd",
}


def escape_popup_text(value):
    return html.escape(clean_html_text(value), quote=True)


def popup_section(title, text):
    text = escape_popup_text(text)
    if not text:
        return ""
    return f"<div class='popup-section'><div class='popup-section-title'>{title}</div><div>{text}</div></div>"


def popup_html(row):
    name = escape_popup_text(row.get("name", "listing")) or "listing"
    room_type = escape_popup_text(row.get("room_type", "unknown")) or "unknown"
    price = escape_popup_text(row.get("price_eur", row.get("price", "")))
    neighbourhood = escape_popup_text(row.get("neighbourhood", ""))

    description = row.get("description_text", "")
    overview = row.get("neighborhood_overview_text", "")
    host_about = row.get("host_about_text", "")

    return f"""
    <div style="width: 360px; max-height: 320px; overflow-y: auto; padding-right: 8px; line-height: 1.35;">
      <style>
        .popup-title {{ font-weight: 700; font-size: 14px; margin-bottom: 6px; }}
        .popup-meta {{ color: #333; margin-bottom: 3px; }}
        .popup-section {{ border-top: 1px solid #ddd; margin-top: 8px; padding-top: 8px; }}
        .popup-section-title {{ font-weight: 700; margin-bottom: 4px; }}
      </style>
      <div class="popup-title">{name}</div>
      <div class="popup-meta"><b>room_type:</b> {room_type}</div>
      <div class="popup-meta"><b>price:</b> {price}</div>
      <div class="popup-meta"><b>neighbourhood:</b> {neighbourhood}</div>
      {popup_section("Description", description)}
      {popup_section("Neighbourhood overview", overview)}
      {popup_section("Host about", host_about)}
    </div>
    """


def map_airbnb_points(gdf, neighbourhoods=None, max_points=1500):
    center = gdf.geometry.unary_union.centroid
    m = folium.Map(location=[center.y, center.x], zoom_start=12, tiles="cartodbpositron")

    if neighbourhoods is not None and len(neighbourhoods):
        folium.GeoJson(
            neighbourhoods[["geometry"]],
            name="Inside Airbnb neighbourhoods",
            style_function=lambda feature: {
                "color": "#444444",
                "weight": 1,
                "fillOpacity": 0.02,
            },
        ).add_to(m)

    plot_gdf = gdf.copy()
    if len(plot_gdf) > max_points:
        plot_gdf = plot_gdf.sample(max_points, random_state=42)

    cluster = MarkerCluster(name=f"Listings sample ({len(plot_gdf):,})").add_to(m)
    for _, row in plot_gdf.iterrows():
        room_type = row.get("room_type", "unknown")
        color = ROOM_TYPE_COLORS.get(room_type, "#d62728")
        folium.CircleMarker(
            location=[row.geometry.y, row.geometry.x],
            radius=4,
            color=color,
            fill=True,
            fill_opacity=0.75,
            popup=folium.Popup(popup_html(row), max_width=420),
        ).add_to(cluster)

    folium.LayerControl().add_to(m)
    return m


MAX_POINTS_ON_MAP = 1500
map_airbnb_points(gdf_listings, gdf_neighbourhoods, max_points=MAX_POINTS_ON_MAP)

## 8. Optional Spatial Join to Neighbourhood Polygons

The CSV already contains a neighbourhood label, but a geometry-based spatial join is still useful to demonstrate the general GIS workflow:

1. take point geometries from `gdf_listings`,
2. take polygon geometries from `gdf_neighbourhoods`,
3. assign polygon attributes to points using `predicate="within"`.

This is the same pattern used later for joining listing points to Zensus grid cells, administrative areas, accessibility zones, or other spatial context layers.

In [ ]:
if len(gdf_neighbourhoods):
    polygon_cols = [col for col in gdf_neighbourhoods.columns if col != "geometry"]
    keep_cols = polygon_cols[:2] + ["geometry"]
    joined = gpd.sjoin(
        gdf_listings,
        gdf_neighbourhoods[keep_cols],
        how="left",
        predicate="within",
    )
    print(f"Spatially joined rows: {len(joined):,}")
    display(joined.head())
else:
    joined = gdf_listings.copy()
    print("No neighbourhood polygons available for spatial join.")

## 9. Save Prepared Point Data

The final GeoJSON contains the cleaned listing table plus point geometry. It is written to Colab's temporary `/content` directory by default.

Use this output when you want to reuse the Munich Airbnb points without repeating the full download and conversion steps in a later notebook.

In [ ]:
out_geojson = CACHE_DIR / "airbnb_munich_listings.geojson"
gdf_listings.to_file(out_geojson, driver="GeoJSON")
out_geojson